# MR1a — Daily Crash MapReduce
**Input** : `mr1/crash_zones_mr_ready.csv`  
**Output** : `mr3/daily_crash_mr.csv`

Schema: `zone_id, crash_date, crashes, injured, killed, avg_lat, avg_lon`

Groups by `(zone_id, crash_date)` — sums crash_count, injury_count, death_count across all hours.

In [17]:
import pandas as pd
import numpy as np
from collections import defaultdict

INPUT  = "crash_zones_mr_ready.csv"
OUTPUT = "daily_crash_mr.csv"

df = pd.read_csv(INPUT)
df["crash_date"]   = pd.to_datetime(df["crash_date"], errors="coerce").dt.strftime("%Y-%m-%d")
df["crash_count"]  = pd.to_numeric(df["crash_count"],  errors="coerce").fillna(0)
df["injury_count"] = pd.to_numeric(df["injury_count"], errors="coerce").fillna(0)
df["death_count"]  = pd.to_numeric(df["death_count"],  errors="coerce").fillna(0)

print(f"Input rows: {len(df):,}")
df.head(10)

Input rows: 2,893


,zone_id,crash_date,crash_hour,crash_count,injury_count,death_count
0,8a2a1008800ffff,2017-02-11,1,1,2.0,0.0
1,8a2a1008800ffff,2019-02-15,20,1,1.0,0.0
2,8a2a1008802ffff,2020-11-05,0,1,0.0,0.0
3,8a2a1008802ffff,2022-10-28,23,1,1.0,0.0
4,8a2a10088047fff,2015-04-01,17,1,0.0,0.0
5,8a2a10088047fff,2020-10-06,20,1,0.0,0.0
6,8a2a10088047fff,2023-10-21,3,1,0.0,0.0
7,8a2a1008804ffff,2015-11-14,6,1,0.0,0.0
8,8a2a1008804ffff,2018-12-04,21,1,1.0,0.0
9,8a2a100880e7fff,2013-11-25,18,1,0.0,0.0


In [14]:
# ── MAP ───────────────────────────────────────────────────────────────────────
# key=(zone_id, crash_date), val=(crash_count, injury_count, death_count)
# avg_lat/avg_lon removed — hex boundaries rendered via zone_id directly.

def mapper(row):
    key = (str(row["zone_id"]), str(row["crash_date"]))
    val = {
        "crashes": row["crash_count"],
        "injured": row["injury_count"],
        "killed":  row["death_count"],
    }
    return key, val

mapped = [mapper(row) for _, row in df.iterrows()]
print(f"Mapped pairs: {len(mapped):,}")

Mapped pairs: 2,893


In [15]:
# ── REDUCE ────────────────────────────────────────────────────────────────────
# Sum crashes/injured/killed per (zone_id, crash_date) across all hours

acc = defaultdict(lambda: {"crashes":0, "injured":0.0, "killed":0.0})

for key, val in mapped:
    a = acc[key]
    a["crashes"] += val["crashes"]
    a["injured"] += val["injured"]
    a["killed"]  += val["killed"]

rows = []
for (zone_id, crash_date), a in acc.items():
    rows.append({
        "zone_id":    zone_id,
        "crash_date": crash_date,
        "crashes":    a["crashes"],
        "injured":    a["injured"],
        "killed":     a["killed"],
    })

out = pd.DataFrame(rows).sort_values(["zone_id","crash_date"]).reset_index(drop=True)
out.to_csv(OUTPUT, index=False)

print(f"Output rows  : {len(out):,}")
print(f"Unique zones : {out['zone_id'].nunique():,}")
print(f"Saved → {OUTPUT}")
out.head()

Output rows  : 2,891
Unique zones : 1,490
Saved → daily_crash_mr.csv


,zone_id,crash_date,crashes,injured,killed
0,8a2a1008800ffff,2017-02-11,1,2.0,0.0
1,8a2a1008800ffff,2019-02-15,1,1.0,0.0
2,8a2a1008802ffff,2020-11-05,1,0.0,0.0
3,8a2a1008802ffff,2022-10-28,1,1.0,0.0
4,8a2a10088047fff,2015-04-01,1,0.0,0.0
